In [15]:
import os
import re
from dotenv import load_dotenv
load_dotenv()

from langchain_gigachat.chat_models import GigaChat

llm = GigaChat(
    credentials=os.getenv("GIGA_KEY"),
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000
)

print("✅ Связь с GigaChat:", llm.invoke("Сколько будет 2+2?").content)

✅ Связь с GigaChat: $2 + 2 = 4$


In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Ты — аналитик заявок на аренду. Извлеки точное количество проживающих.
Правила:
- "пара", "двое", "молодая семья" = 2
- "семья с N детьми" = 2 + N
- "студент", "один", "индивидуально" = 1
- Числительные ("троих", "четверых") переводи в цифры.
Верни ТОЛЬКО целое число без пояснений.

Заявка: {text}
Количество:"""
)

chain = basic_prompt | llm | StrOutputParser()

In [17]:
# Ваши данные из rental_12.csv
import re
test_texts = [
    "ищем жилье в центре недалеко от моря с 23.07-03.08 - нужен 1 двухместный номер, 1 трехместный, недорого. или как вариант дом на 2 семьи (5 чел)",
    "2 семьи по 3 человека (2 взрослых и ребенок) с 01.09-13.09 снимем жильё в пределах 5 минут от пляжа, 1500 руб/сутки. Рассмотрим все варианты.",
    "Добрый день! Ищем жильё (гостевой дом) с парковкой в Лазаревском, планируем приезд 5-6 августа на 7 дней, двое взрослых и двое детей (16 и 2 г) не далеко от моря. Рассмотрим ваши предложения!",
    "Здравствуйте ! Ищем жилье в Лазаревском 4 человека, с бассейном с 1.08 по 12.08!",
    "Ищем жилье в самом Лазаревском для двоих, 2-местный номер с удобствами, с 7-8 сентября не менее чем на неделю, 700 р. за номер",
    "Интересует жилье на одного человека с 08.07. до 15.07. Не далеко от пляжа 🤗",
    "Ищем жильё для 2-х взрослых, ребёнок (7 лет) и собака. Только Чёрное море, не Крым и Анапа. С 15 июля на 10 дней. Предложения в личку.",
    "Интересует жилье в Сочи у пляжа Ривьера (максимально близко к морю!!!) с 8 по 15 июля. 1 номер/квартира/комната, 3 человека. Предложения в л/с",
    "Лазаревское, Солоники - бюджетное жилье на 3 суток 4 взрослых с 5.09 интересует. Только в личку.",
    "Здравствуйте, ищем жилье с 10.08по 20.08,два взрослых,два ребенка,не далеко от моря.",
    "Ищем жилье в Лазаревском с 12 по 18 августа. Не дороже 1000р за комнату. Нас двое.",
    "Ищем жильё эконом класса, до 1000 на двоих, с 7 по 15 августа, недалеко от моря. Писать в личку",
    "Здравствуйте🤗 Ищем жильё в центре рядом с морем🌊 Цена за номер до2️⃣0️⃣0️⃣0️⃣руб с удобствами 🚿 Приезжаем парой с 10 августа по 18 августа 📆",
    "Здравствуйте!Ищем жильё в Лазаревское.2 взрослых и 1 ребёнок 10 лет.В период с 25.07. По 06.08.",
    "ищем жилье с 8-20 августа с бассейном 3мест номер ,до моря не дольше 10мин, не в гору"
]

print("🧪 Тест базовой цепочки:")
for i, t in enumerate(test_texts, 1):
    raw = chain.invoke({"text": t})
    nums = re.findall(r'\d+', raw)
    cleaned = nums[0] if nums else "❌"
    print(f"{i}. {cleaned} ← {t[:60]}...")

🧪 Тест базовой цепочки:
1. 5 ← ищем жилье в центре недалеко от моря с 23.07-03.08 - нужен 1...
2. 8 ← 2 семьи по 3 человека (2 взрослых и ребенок) с 01.09-13.09 с...
3. 4 ← Добрый день! Ищем жильё (гостевой дом) с парковкой в Лазарев...
4. 4 ← Здравствуйте ! Ищем жилье в Лазаревском 4 человека, с бассей...
5. 2 ← Ищем жилье в самом Лазаревском для двоих, 2-местный номер с ...
6. 1 ← Интересует жилье на одного человека с 08.07. до 15.07. Не да...
7. 3 ← Ищем жильё для 2-х взрослых, ребёнок (7 лет) и собака. Тольк...
8. 3 ← Интересует жилье в Сочи у пляжа Ривьера (максимально близко ...
9. 4 ← Лазаревское, Солоники - бюджетное жилье на 3 суток 4 взрослы...
10. 4 ← Здравствуйте, ищем жилье с 10.08по 20.08,два взрослых,два ре...
11. 2 ← Ищем жилье в Лазаревском с 12 по 18 августа. Не дороже 1000р...
12. 2 ← Ищем жильё эконом класса, до 1000 на двоих, с 7 по 15 август...
13. 2 ← Здравствуйте🤗 Ищем жильё в центре рядом с морем🌊 Цена за ном...
14. 3 ← Здравствуйте!Ищем жильё в Лазаревское.2 в